# 05 - Results Analysis

Final performance evaluation of the DenseNet-Attention model on the held-out test set.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import get_dataloaders
from src.models import get_model
from src.evaluate import evaluate_model, plot_confusion_matrix
from src.utils import set_seed, get_device, load_config

plt.rcParams["figure.dpi"] = 130
sns.set_style("whitegrid")

config = load_config("../configs/default.yaml")
device = get_device()
DATA_ROOT = "../data/chest_xray"
SEED = config["reproducibility"]["seed"]

print(f"Device: {device}")

In [ ]:
set_seed(SEED)

dataloaders = get_dataloaders(
    DATA_ROOT,
    augmentation="standard",
    image_size=config["data"]["image_size"],
    batch_size=32,
    num_workers=0,
    seed=SEED
)

model = get_model("densenet_attention", pretrained=True, use_attention=True)
checkpoint_path = "../results/checkpoints/densenet_attention_best.pt"

model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model = model.to(device)
model.eval()

print("✅ Best DenseNet-Attention model loaded successfully.")

In [ ]:
test_metrics = evaluate_model(model, dataloaders["test"], device)

print("=== FINAL TEST RESULTS ===")
print(f"AUROC          : {test_metrics['auroc']:.4f}")
print(f"F1 (macro)     : {test_metrics['f1_macro']:.4f}")
print(f"Accuracy       : {test_metrics['accuracy']:.4f}")
print(f"Sensitivity    : {test_metrics['sensitivity']:.4f}")
print(f"Specificity    : {test_metrics['specificity']:.4f}")
print(f"NPV            : {test_metrics['npv']:.4f}")

In [ ]:
plot_confusion_matrix(test_metrics["confusion_matrix"], class_names=["NORMAL", "PNEUMONIA"])
plt.show()

## Summary of Results

- Strong performance on the pediatric chest X-ray pneumonia dataset
- Clear improvement over ResNet-18 baseline
- Model is ready for clinical triage / flagging support use case